In [0]:
%run ./connectionNotebook

In [0]:
catalog_name = 'adbrag'
target_schema_name = 'gold'
src_schema_name = 'silver'

In [0]:
from pyspark.sql.functions import col, when, current_timestamp

silver_df = spark.table(f"{catalog_name}.{src_schema_name}.products_silver")

gold_df = (
    silver_df
    .withColumn(
        "price_band",
        when(col("src_Price") < 100, "Low")
        .when((col("src_Price") >= 100) & (col("src_Price") < 300), "Medium")
        .otherwise("High")
    )
    .withColumn(
        "product_status",
        when(col("src_Price") > 0, "Active").otherwise("Inactive")
    )
    .withColumn("gold_loaded_ts", current_timestamp())
)



In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{target_schema_name}.products_gold
(
    src_ProductID INT,
    src_ProductName STRING,
    src_Category STRING,
    src_Price DECIMAL(10,2),
    src_CreatedDate TIMESTAMP,
    processed_ts TIMESTAMP,
    price_band STRING,
    product_status STRING,
    gold_loaded_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (src_ProductID)
""")



In [0]:
gold_df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog_name}.{target_schema_name}.products_gold")